# Reconhecimento de Polímeros Plásticos - Benchmark de 8 Modelos no Google Colab
Este notebook está dividido em células independentes para comparar **2 Estratégias de Divisão de Dados** em **4 Arquiteturas de IA**:

### Grupo 1: Estratégia LOOO (Leave-One-Object-Out / Group Split)
Garante que garrafas plásticas inteiras fiquem restritas ao treino ou à validação (sem vazamento de dados).
1. **CustomCNN LOOO** (`modelo_cnn_looo.pth`)
2. **ResNet-50 LOOO** (`modelo_resnet_looo.pth`)
3. **ConvNeXt-Tiny LOOO** (`modelo_convnext_looo.pth`)
4. **Swin Transformer LOOO** (`modelo_swin_looo.pth`)

### Grupo 2: Estratégia Random Split (Separação Aleatória de Imagens)
Divisão aleatória onde frames do mesmo objeto podem estar no treino e na validação (demonstração de Data Leakage).
5. **CustomCNN Random** (`modelo_cnn_random.pth`)
6. **ResNet-50 Random** (`modelo_resnet_random.pth`)
7. **ConvNeXt-Tiny Random** (`modelo_convnext_random.pth`)
8. **Swin Transformer Random** (`modelo_swin_random.pth`)

In [ ]:
# =====================================================================
# CÉLULA 1: Montar Google Drive e Descompactar Dataset
# =====================================================================
from google.colab import drive
import os
import zipfile

drive.mount('/content/drive')

DATASET_ZIP_PATH = '/content/drive/MyDrive/Dataset_Wadaba.zip'
DATA_DIR = './Dataset_Wadaba'

if os.path.exists(DATASET_ZIP_PATH):
    if not os.path.exists(DATA_DIR):
        print(f'Descompactando {DATASET_ZIP_PATH} para {DATA_DIR}...')
        with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print('Descompactação concluída!')
    else:
        print('Dataset já descompactado localmente.')
else:
    print(f'Aviso: Zip não encontrado em {DATASET_ZIP_PATH}. Verifique o caminho!')

In [ ]:
# =====================================================================
# CÉLULA 2: Imports, GPU (CUDA) e Carregadores (LOOO vs Random Split)
# =====================================================================
import re
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from collections import defaultdict

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de Execução Ativo: {device}')
if torch.cuda.is_available():
    print(f'GPU Detectada: {torch.cuda.get_device_name(0)}')

class WaDaBaDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, torch.tensor(label, dtype=torch.long)

def load_all_paths(data_dir):
    class_names = ['Other', 'PET', 'PE_HD', 'PP', 'PS']
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    code_to_class = {'01': 'PET', '02': 'PE_HD', '05': 'PP', '06': 'PS', '07': 'Other'}
    img_paths, labels = [], []
    object_images = defaultdict(lambda: defaultdict(list))
    for root, dirs, files in os.walk(data_dir):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                path = os.path.join(root, f)
                filename = os.path.basename(path)
                parent_dir = os.path.basename(os.path.dirname(path))
                class_name = parent_dir if parent_dir in class_to_idx else code_to_class.get(re.search(r'_a(\d{2})', filename).group(1) if re.search(r'_a(\d{2})', filename) else '', 'Other')
                match_obj = re.match(r'^(\d+)', filename)
                obj_id = match_obj.group(1) if match_obj else filename
                object_images[class_name][obj_id].append(path)
                img_paths.append(path)
                labels.append(class_to_idx[class_name])
    return img_paths, labels, object_images, class_names, class_to_idx

def get_transforms():
    train_tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(72),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.RandomPerspective(distortion_scale=0.1, p=0.5),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.2), ratio=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_tf = transforms.Compose([
        transforms.Resize(IMG_SIZE), transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_tf, val_tf

def create_datasets_looo(data_dir, val_split=0.2, seed=123):
    img_paths, labels, object_images, class_names, class_to_idx = load_all_paths(data_dir)
    train_paths, train_labels, val_paths, val_labels = [], [], [], []
    rng = np.random.default_rng(seed)
    for class_name in class_names:
        class_idx = class_to_idx[class_name]
        objects_dict = object_images[class_name]
        unique_obj_ids = sorted(list(objects_dict.keys()))
        if not unique_obj_ids: continue
        shuffled_objs = unique_obj_ids.copy()
        rng.shuffle(shuffled_objs)
        split_idx = max(1, int(len(shuffled_objs) * (1.0 - val_split)))
        for obj_id in shuffled_objs[:split_idx]:
            for path in objects_dict[obj_id]: train_paths.append(path); train_labels.append(class_idx)
        for obj_id in shuffled_objs[split_idx:]:
            for path in objects_dict[obj_id]: val_paths.append(path); val_labels.append(class_idx)
    train_tf, val_tf = get_transforms()
    return WaDaBaDataset(train_paths, train_labels, train_tf), WaDaBaDataset(val_paths, val_labels, val_tf), class_names

def create_datasets_random(data_dir, val_split=0.2, seed=123):
    img_paths, labels, _, class_names, _ = load_all_paths(data_dir)
    train_paths, val_paths, train_labels, val_labels = train_test_split(img_paths, labels, test_size=val_split, random_state=seed, stratify=labels)
    train_tf, val_tf = get_transforms()
    return WaDaBaDataset(train_paths, train_labels, train_tf), WaDaBaDataset(val_paths, val_labels, val_tf), class_names

In [ ]:
# =====================================================================
# CÉLULA 3: Construtores de Arquitetura e Funções de Treino
# =====================================================================
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(256 * 14 * 14, 128), nn.ReLU(), nn.Linear(128, num_classes))
    def forward(self, x): return self.classifier(self.features(x))

def build_resnet_model(num_classes):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    for param in model.parameters(): param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_convnext_model(num_classes):
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    for param in model.parameters(): param.requires_grad = False
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    return model

def build_swin_model(num_classes):
    model = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)
    for param in model.parameters(): param.requires_grad = False
    model.head = nn.Linear(model.head.in_features, num_classes)
    return model

class EarlyStopping:
    def __init__(self, patience=8):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_weights = None
    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
    def restore(self, model):
        if self.best_weights: model.load_state_dict(self.best_weights)

def plot_history(history, model_name):
    epochs = range(1, len(history['accuracy']) + 1)
    plt.figure(figsize=(12, 4.5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['accuracy'], 'b-o', label='Acurácia Treino')
    plt.plot(epochs, history['val_accuracy'], 'r-o', label='Acurácia Validação')
    plt.title(f'Acurácia vs Épocas ({model_name})')
    plt.xlabel('Época')
    plt.ylabel('Acurácia')
    plt.grid(True)
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['loss'], 'b-o', label='Perda (Loss) Treino')
    plt.plot(epochs, history['val_loss'], 'r-o', label='Perda (Loss) Validação')
    plt.title(f'Perda (Loss) vs Épocas ({model_name})')
    plt.xlabel('Época')
    plt.ylabel('Perda (Loss)')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{model_name}_history.png')
    plt.show()
    plt.close()

def train_and_eval(model, train_ds, val_ds, class_names, model_name, lr=1e-3, weight_decay=0.0):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    y_train = train_ds.labels
    weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    early_stopping = EarlyStopping(patience=8)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=3)
    
    history = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
    for epoch in range(EPOCHS):
        model.train()
        r_loss, r_corr, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            _, preds = torch.max(outputs, 1)
            r_loss += loss.item() * inputs.size(0)
            r_corr += torch.sum(preds == labels)
            total += inputs.size(0)
        
        model.eval()
        v_loss, v_corr, v_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                v_loss += criterion(outputs, labels).item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                v_corr += torch.sum(preds == labels)
                v_total += inputs.size(0)
        val_loss, val_acc = v_loss / v_total, (v_corr.double() / v_total).item()
        train_loss, train_acc = r_loss / total, (r_corr.double() / total).item()
        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'Epoch {epoch+1:02d}/{EPOCHS:02d} - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')
        scheduler.step(val_loss)
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print('Early stopping ativado!')
            early_stopping.restore(model)
            break
    
    torch.save(model.state_dict(), f'{model_name}.pth')
    plot_history(history, model_name)
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs.to(device))
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.numpy()); y_pred.extend(preds.cpu().numpy())
    print(f'\n--- Relatório Final: {model_name} ---')
    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    print(report)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Previsto')
    plt.ylabel('Real')
    plt.title(f'Matriz de Confusão - {model_name}')
    plt.tight_layout()
    plt.savefig(f'{model_name}_confusion_matrix.png')
    plt.show()
    plt.close()
    return (np.array(y_true) == np.array(y_pred)).mean()

In [ ]:
# =====================================================================
# MODELO 1: CustomCNN LOOO (Group Split por Objeto)
# =====================================================================
train_ds_looo, val_ds_looo, class_names = create_datasets_looo(DATA_DIR)
cnn_looo = CustomCNN(len(class_names)).to(device)
acc_cnn_looo = train_and_eval(cnn_looo, train_ds_looo, val_ds_looo, class_names, 'modelo_cnn_looo', lr=1e-3)

In [ ]:
# =====================================================================
# MODELO 2: ResNet-50 LOOO (Group Split por Objeto)
# =====================================================================
resnet_looo = build_resnet_model(len(class_names)).to(device)
acc_resnet_looo = train_and_eval(resnet_looo, train_ds_looo, val_ds_looo, class_names, 'modelo_resnet_looo', lr=1e-3, weight_decay=0.01)

In [ ]:
# =====================================================================
# MODELO 3: ConvNeXt-Tiny LOOO (Group Split por Objeto)
# =====================================================================
convnext_looo = build_convnext_model(len(class_names)).to(device)
acc_convnext_looo = train_and_eval(convnext_looo, train_ds_looo, val_ds_looo, class_names, 'modelo_convnext_looo', lr=5e-4, weight_decay=0.01)

In [ ]:
# =====================================================================
# MODELO 4: Swin Transformer LOOO (Group Split por Objeto)
# =====================================================================
swin_looo = build_swin_model(len(class_names)).to(device)
acc_swin_looo = train_and_eval(swin_looo, train_ds_looo, val_ds_looo, class_names, 'modelo_swin_looo', lr=5e-4, weight_decay=0.01)

In [ ]:
# =====================================================================
# MODELO 5: CustomCNN RANDOM (Separação Aleatória por Imagem)
# =====================================================================
train_ds_rnd, val_ds_rnd, _ = create_datasets_random(DATA_DIR)
cnn_rnd = CustomCNN(len(class_names)).to(device)
acc_cnn_rnd = train_and_eval(cnn_rnd, train_ds_rnd, val_ds_rnd, class_names, 'modelo_cnn_random', lr=1e-3)

In [ ]:
# =====================================================================
# MODELO 6: ResNet-50 RANDOM (Separação Aleatória por Imagem)
# =====================================================================
resnet_rnd = build_resnet_model(len(class_names)).to(device)
acc_resnet_rnd = train_and_eval(resnet_rnd, train_ds_rnd, val_ds_rnd, class_names, 'modelo_resnet_random', lr=1e-3, weight_decay=0.01)

In [ ]:
# =====================================================================
# MODELO 7: ConvNeXt-Tiny RANDOM (Separação Aleatória por Imagem)
# =====================================================================
convnext_rnd = build_convnext_model(len(class_names)).to(device)
acc_convnext_rnd = train_and_eval(convnext_rnd, train_ds_rnd, val_ds_rnd, class_names, 'modelo_convnext_random', lr=5e-4, weight_decay=0.01)

In [ ]:
# =====================================================================
# MODELO 8: Swin Transformer RANDOM (Separação Aleatória por Imagem)
# =====================================================================
swin_rnd = build_swin_model(len(class_names)).to(device)
acc_swin_rnd = train_and_eval(swin_rnd, train_ds_rnd, val_ds_rnd, class_names, 'modelo_swin_random', lr=5e-4, weight_decay=0.01)